In [53]:
!pip install streamlit plotly pandas pyngrok -q
print("✅ All libraries installed!")

✅ All libraries installed!


In [54]:
from google.colab import files
uploaded = files.upload()
# Select Nassau_Candy.csv → wait for 100% ✅

Saving Nassau_Candy.csv to Nassau_Candy (1).csv


In [55]:
import pandas as pd, os, glob

# Auto-find your CSV
csv_files = glob.glob("/content/*.csv")
print("📁 Files found:", csv_files)

df_test = pd.read_csv(csv_files[0])
print("\n✅ Columns:", df_test.columns.tolist())
print(f"✅ Rows: {len(df_test):,}")
print(df_test.head(3))

# Save exact path for next cells
CSV_PATH = csv_files[0]
print(f"\n📌 Your file path: {CSV_PATH}")

📁 Files found: ['/content/Nassau_Candy (1).csv', '/content/Nassau_Candy.csv']

✅ Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Country/Region', 'City', 'State/Province', 'Postal Code', 'Division', 'Region', 'Product ID', 'Product Name', 'Sales', 'Units', 'Gross Profit', 'Cost']
✅ Rows: 10,194
   Row ID                      Order ID  Order Date   Ship Date  \
0       1  US-2021-103800-CHO-MIL-31000  03-01-2024  30-06-2024   
1       2  US-2021-112326-CHO-TRI-54000  04-01-2024  01-07-2024   
2       3  US-2021-112326-CHO-NUT-13000  04-01-2024  01-07-2024   

        Ship Mode  Customer ID Country/Region        City State/Province  \
0  Standard Class       103800  United States     Houston          Texas   
1  Standard Class       112326  United States  Naperville       Illinois   
2  Standard Class       112326  United States  Naperville       Illinois   

  Postal Code   Division    Region     Product ID  \
0       77095  Chocolate  Interior  C

In [56]:
%%writefile /content/app.py

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import glob

# ─────────────────────────────────────────────
# PAGE CONFIG
# ─────────────────────────────────────────────
st.set_page_config(
    page_title="Nassau Candy | Route Efficiency",
    page_icon="🍬",
    layout="wide"
)

# ─────────────────────────────────────────────
# STYLING
# ─────────────────────────────────────────────
st.markdown("""
<style>
[data-testid="stAppViewContainer"] { background-color: #0f0f1a; }
[data-testid="stSidebar"]          { background-color: #12122a; }
[data-testid="stHeader"]           { background-color: #0f0f1a; }
h1, h2, h3, h4                     { color: #f5a623; }
p, label, div                       { color: #e0e0e0; }
div[data-testid="metric-container"] {
    background-color: #1e1e3a;
    border: 1px solid #f5a623;
    border-radius: 12px;
    padding: 12px;
    text-align: center;
}
div[data-testid="metric-container"] label { color: #f5a623 !important; font-size: 13px; }
div[data-testid="metric-container"] div   { color: #ffffff !important; font-size: 22px; font-weight: bold; }
.stTabs [data-baseweb="tab"]        { color: #f5a623; font-weight: bold; }
.stTabs [aria-selected="true"]      { border-bottom: 3px solid #f5a623; }
</style>
""", unsafe_allow_html=True)

# ─────────────────────────────────────────────
# LOAD & CLEAN DATA
# ─────────────────────────────────────────────
@st.cache_data
def load_data():
    csv_files = glob.glob("/content/*.csv")
    df = pd.read_csv(csv_files[0])

    df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
    df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  dayfirst=True, errors="coerce")
    df["Lead Time"]  = (df["Ship Date"] - df["Order Date"]).dt.days
    df = df[df["Lead Time"] >= 0].dropna(subset=["Lead Time", "Order Date", "Ship Date"])

    factory_map = {
        "Wonka Bar - Nutty Crunch Surprise":  "Lot's O Nuts",
        "Wonka Bar - Fudge Mallows":          "Lot's O Nuts",
        "Wonka Bar -Scrumdiddlyumptious":     "Lot's O Nuts",
        "Wonka Bar - Milk Chocolate":         "Wicked Choccy's",
        "Wonka Bar - Triple Dazzle Caramel":  "Wicked Choccy's",
        "Laffy Taffy":                        "Sugar Shack",
        "sweeTARTS":                          "Sugar Shack",
        "Nerds":                              "Sugar Shack",
        "Fun Dip":                            "Sugar Shack",
        "Fizzy Lifting Drinks":               "Sugar Shack",
        "Everlasting Gobstopper":             "Secret Factory",
        "Hair Toffee":                        "Secret Factory",
        "Lickable Wallpaper":                 "Secret Factory",
        "Wonka Gum":                          "The Other Factory",
        "Kazookles":                          "The Other Factory",
    }
    df["Factory"]    = df["Product Name"].map(factory_map).fillna("Unknown")
    df["Route"]      = df["Factory"] + " → " + df["State/Province"]
    df["Is Delayed"] = df["Lead Time"] > 5
    df["Month"]      = df["Order Date"].dt.to_period("M").astype(str)
    df["Year"]       = df["Order Date"].dt.year.astype(str)

    # Route Efficiency Score (lower lead time = higher score)
    max_lt = df["Lead Time"].max()
    df["Efficiency Score"] = ((max_lt - df["Lead Time"]) / max_lt * 100).round(1)

    return df

df = load_data()

DARK = dict(
    paper_bgcolor="#0f0f1a", plot_bgcolor="#0f0f1a",
    font=dict(color="#e0e0e0"),
    title_font=dict(color="#f5a623", size=15),
    margin=dict(l=10, r=10, t=50, b=10),
    legend=dict(bgcolor="#1e1e3a", bordercolor="#f5a623", borderwidth=1)
)

# ─────────────────────────────────────────────
# HEADER
# ─────────────────────────────────────────────
st.markdown("""
<div style='text-align:center; padding: 20px 0 10px 0;'>
  <h1>🍬 Nassau Candy Distributor</h1>
  <h4 style='color:#cccccc;'>Factory-to-Customer Shipping Route Efficiency Dashboard</h4>
</div>
""", unsafe_allow_html=True)
st.markdown("---")

# ─────────────────────────────────────────────
# SIDEBAR FILTERS
# ─────────────────────────────────────────────
st.sidebar.markdown("## 🔧 Dashboard Filters")

min_date   = df["Order Date"].min().date()
max_date   = df["Order Date"].max().date()
date_range = st.sidebar.date_input("📅 Order Date Range",
                                    [min_date, max_date],
                                    min_value=min_date,
                                    max_value=max_date)

regions    = ["All"] + sorted(df["Region"].dropna().unique().tolist())
ship_modes = ["All"] + sorted(df["Ship Mode"].dropna().unique().tolist())
factories  = ["All"] + sorted(df["Factory"].dropna().unique().tolist())
divisions  = ["All"] + sorted(df["Division"].dropna().unique().tolist())

region    = st.sidebar.selectbox("🌍 Region",    regions)
ship_mode = st.sidebar.selectbox("🚚 Ship Mode", ship_modes)
factory   = st.sidebar.selectbox("🏭 Factory",   factories)
division  = st.sidebar.selectbox("📦 Division",  divisions)
max_lead  = st.sidebar.slider("⏱️ Max Lead Time (days)",
                               0, int(df["Lead Time"].max()),
                               int(df["Lead Time"].max()))

st.sidebar.markdown("---")
st.sidebar.markdown("### 🚨 Delay Threshold")
delay_threshold = st.sidebar.slider("Flag orders delayed beyond (days)", 1, 30, 5)

# ─────────────────────────────────────────────
# APPLY FILTERS
# ─────────────────────────────────────────────
fdf = df.copy()
fdf["Is Delayed"] = fdf["Lead Time"] > delay_threshold

if len(date_range) == 2:
    fdf = fdf[(fdf["Order Date"].dt.date >= date_range[0]) &
              (fdf["Order Date"].dt.date <= date_range[1])]
if region    != "All": fdf = fdf[fdf["Region"]    == region]
if ship_mode != "All": fdf = fdf[fdf["Ship Mode"] == ship_mode]
if factory   != "All": fdf = fdf[fdf["Factory"]   == factory]
if division  != "All": fdf = fdf[fdf["Division"]  == division]
fdf = fdf[fdf["Lead Time"] <= max_lead]

# ─────────────────────────────────────────────
# KPI CARDS
# ─────────────────────────────────────────────
st.markdown("### 📊 Key Performance Indicators")
k1,k2,k3,k4,k5,k6 = st.columns(6)
k1.metric("📦 Total Orders",    f"{len(fdf):,}")
k2.metric("⏱️ Avg Lead Time",   f"{fdf['Lead Time'].mean():.1f} days")
k3.metric("🚨 Delay Rate",      f"{fdf['Is Delayed'].mean()*100:.1f}%")
k4.metric("💰 Total Sales",     f"${fdf['Sales'].sum():,.0f}")
k5.metric("📈 Gross Profit",    f"${fdf['Gross Profit'].sum():,.0f}")
k6.metric("🛣️ Active Routes",   f"{fdf['Route'].nunique():,}")
st.markdown("---")

# ─────────────────────────────────────────────
# TABS
# ─────────────────────────────────────────────
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    "🛣️ Route Efficiency",
    "🗺️ Geographic Map",
    "🚚 Ship Mode Analysis",
    "📈 Trends",
    "📋 Order Detail"
])

# ══════════════════════════════════════════════
# TAB 1 — ROUTE EFFICIENCY
# ══════════════════════════════════════════════
with tab1:
    st.markdown("### 🛣️ Route Efficiency Overview")

    route_agg = fdf.groupby("Route").agg(
        Avg_Lead       =("Lead Time", "mean"),
        Shipments      =("Lead Time", "count"),
        Delay_Rate     =("Is Delayed", "mean"),
        Avg_Efficiency =("Efficiency Score", "mean")
    ).reset_index()
    route_agg["Avg_Lead"]       = route_agg["Avg_Lead"].round(1)
    route_agg["Delay_Rate"]     = (route_agg["Delay_Rate"] * 100).round(1)
    route_agg["Avg_Efficiency"] = route_agg["Avg_Efficiency"].round(1)

    col1, col2 = st.columns(2)

    with col1:
        top10 = route_agg.nsmallest(10, "Avg_Lead")
        fig = px.bar(top10, x="Avg_Lead", y="Route", orientation="h",
                     color="Avg_Lead", color_continuous_scale="Greens_r",
                     title="✅ Top 10 Most Efficient Routes (Lowest Lead Time)",
                     labels={"Avg_Lead": "Avg Lead Time (days)", "Route": "Route"},
                     text="Avg_Lead")
        fig.update_traces(texttemplate="%{text:.1f}d", textposition="outside")
        fig.update_layout(DARK, height=420)
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        bot10 = route_agg.nlargest(10, "Avg_Lead")
        fig2 = px.bar(bot10, x="Avg_Lead", y="Route", orientation="h",
                      color="Avg_Lead", color_continuous_scale="Reds",
                      title="🚨 Bottom 10 Least Efficient Routes (Highest Lead Time)",
                      labels={"Avg_Lead": "Avg Lead Time (days)", "Route": "Route"},
                      text="Avg_Lead")
        fig2.update_traces(texttemplate="%{text:.1f}d", textposition="outside")
        fig2.update_layout(DARK, height=420)
        st.plotly_chart(fig2, use_container_width=True)

    st.markdown("### 🏆 Route Performance Leaderboard")
    leaderboard = route_agg.sort_values("Avg_Lead").reset_index(drop=True)
    leaderboard.index += 1
    leaderboard.columns = ["Route","Avg Lead (days)","Total Shipments","Delay Rate (%)","Efficiency Score"]
    st.dataframe(leaderboard, use_container_width=True, height=350)

# ══════════════════════════════════════════════
# TAB 2 — GEOGRAPHIC MAP
# ══════════════════════════════════════════════
with tab2:
    st.markdown("### 🗺️ Geographic Shipping Efficiency")

    state_agg = fdf.groupby("State/Province").agg(
        Avg_Lead  =("Lead Time","mean"),
        Orders    =("Lead Time","count"),
        Delay_Rate=("Is Delayed","mean")
    ).reset_index()
    state_agg.columns = ["State","Avg Lead Time","Orders","Delay Rate"]
    state_agg["Delay Rate"] = (state_agg["Delay Rate"]*100).round(1)
    state_agg["Avg Lead Time"] = state_agg["Avg Lead Time"].round(1)

    col1, col2 = st.columns(2)

    with col1:
        fig_map1 = px.choropleth(state_agg,
                                  locations="State",
                                  locationmode="USA-states",
                                  color="Avg Lead Time",
                                  scope="usa",
                                  color_continuous_scale="RdYlGn_r",
                                  hover_data=["Orders","Delay Rate"],
                                  title="🗺️ Avg Lead Time by State (Red = Slow, Green = Fast)")
        fig_map1.update_layout(DARK, height=420)
        st.plotly_chart(fig_map1, use_container_width=True)

    with col2:
        fig_map2 = px.choropleth(state_agg,
                                  locations="State",
                                  locationmode="USA-states",
                                  color="Delay Rate",
                                  scope="usa",
                                  color_continuous_scale="Reds",
                                  hover_data=["Orders","Avg Lead Time"],
                                  title="🚨 Delay Rate by State (%)")
        fig_map2.update_layout(DARK, height=420)
        st.plotly_chart(fig_map2, use_container_width=True)

    st.markdown("### 🏙️ Regional Bottleneck Analysis")
    region_agg = fdf.groupby("Region").agg(
        Avg_Lead  =("Lead Time","mean"),
        Orders    =("Lead Time","count"),
        Delay_Rate=("Is Delayed","mean")
    ).reset_index()
    region_agg["Delay_Rate"] = (region_agg["Delay_Rate"]*100).round(1)

    fig_reg = px.scatter(region_agg,
                          x="Avg_Lead", y="Delay_Rate",
                          size="Orders", color="Region",
                          title="📍 Regional Bottleneck Map (Bubble Size = Volume)",
                          labels={"Avg_Lead":"Avg Lead Time (days)","Delay_Rate":"Delay Rate (%)"},
                          text="Region")
    fig_reg.update_traces(textposition="top center")
    fig_reg.update_layout(DARK, height=400)
    st.plotly_chart(fig_reg, use_container_width=True)

# ══════════════════════════════════════════════
# TAB 3 — SHIP MODE ANALYSIS
# ══════════════════════════════════════════════
with tab3:
    st.markdown("### 🚚 Ship Mode Performance Analysis")

    col1, col2 = st.columns(2)

    with col1:
        mode_df = fdf["Ship Mode"].value_counts().reset_index()
        mode_df.columns = ["Ship Mode","Count"]
        fig_pie = px.pie(mode_df, names="Ship Mode", values="Count",
                         title="🚚 Orders by Ship Mode",
                         hole=0.45,
                         color_discrete_sequence=px.colors.qualitative.Set2)
        fig_pie.update_layout(DARK, height=380)
        st.plotly_chart(fig_pie, use_container_width=True)

    with col2:
        mode_lead = fdf.groupby("Ship Mode")["Lead Time"].mean().reset_index()
        mode_lead.columns = ["Ship Mode","Avg Lead Time"]
        fig_mode = px.bar(mode_lead, x="Ship Mode", y="Avg Lead Time",
                          color="Ship Mode", title="⏱️ Avg Lead Time by Ship Mode",
                          color_discrete_sequence=px.colors.qualitative.Pastel,
                          text="Avg Lead Time")
        fig_mode.update_traces(texttemplate="%{text:.1f}d", textposition="outside")
        fig_mode.update_layout(DARK, height=380)
        st.plotly_chart(fig_mode, use_container_width=True)

    st.markdown("---")
    col3, col4 = st.columns(2)

    with col3:
        mode_delay = fdf.groupby("Ship Mode")["Is Delayed"].mean().reset_index()
        mode_delay["Delay Rate (%)"] = (mode_delay["Is Delayed"]*100).round(1)
        fig_delay = px.bar(mode_delay, x="Ship Mode", y="Delay Rate (%)",
                           color="Ship Mode", title="🚨 Delay Rate by Ship Mode (%)",
                           color_discrete_sequence=px.colors.qualitative.Set1,
                           text="Delay Rate (%)")
        fig_delay.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
        fig_delay.update_layout(DARK, height=350)
        st.plotly_chart(fig_delay, use_container_width=True)

    with col4:
        mode_sales = fdf.groupby("Ship Mode").agg(
            Sales       =("Sales","sum"),
            Gross_Profit=("Gross Profit","sum")
        ).reset_index()
        fig_cost = px.bar(mode_sales, x="Ship Mode",
                          y=["Sales","Gross_Profit"],
                          title="💰 Sales vs Gross Profit by Ship Mode",
                          barmode="group",
                          color_discrete_sequence=["#f5a623","#00cc96"])
        fig_cost.update_layout(DARK, height=350)
        st.plotly_chart(fig_cost, use_container_width=True)

# ══════════════════════════════════════════════
# TAB 4 — TRENDS
# ══════════════════════════════════════════════
with tab4:
    st.markdown("### 📈 Shipping Performance Trends Over Time")

    trend = fdf.groupby("Month").agg(
        Avg_Lead  =("Lead Time","mean"),
        Orders    =("Lead Time","count"),
        Delay_Rate=("Is Delayed","mean")
    ).reset_index()
    trend["Delay_Rate"] = (trend["Delay_Rate"]*100).round(1)

    fig_line = px.line(trend, x="Month", y="Avg_Lead",
                       title="📈 Avg Lead Time Trend (Monthly)",
                       markers=True,
                       color_discrete_sequence=["#f5a623"],
                       labels={"Avg_Lead":"Avg Lead Time (days)"})
    fig_line.update_layout(DARK, height=350)
    st.plotly_chart(fig_line, use_container_width=True)

    col1, col2 = st.columns(2)
    with col1:
        fig_vol = px.bar(trend, x="Month", y="Orders",
                         title="📦 Monthly Order Volume",
                         color_discrete_sequence=["#636efa"])
        fig_vol.update_layout(DARK, height=320)
        st.plotly_chart(fig_vol, use_container_width=True)

    with col2:
        fig_dr = px.line(trend, x="Month", y="Delay_Rate",
                         title="🚨 Monthly Delay Rate (%)",
                         markers=True, color_discrete_sequence=["#ef553b"],
                         labels={"Delay_Rate":"Delay Rate (%)"})
        fig_dr.update_layout(DARK, height=320)
        st.plotly_chart(fig_dr, use_container_width=True)

    st.markdown("### 🏭 Factory Performance Over Time")
    fac_trend = fdf.groupby(["Month","Factory"])["Lead Time"].mean().reset_index()
    fig_fac = px.line(fac_trend, x="Month", y="Lead Time",
                      color="Factory",
                      title="🏭 Avg Lead Time by Factory (Monthly)",
                      markers=True)
    fig_fac.update_layout(DARK, height=380)
    st.plotly_chart(fig_fac, use_container_width=True)

# ══════════════════════════════════════════════
# TAB 5 — ORDER DETAIL TABLE
# ══════════════════════════════════════════════
with tab5:
    st.markdown("### 📋 Order-Level Shipment Timeline")

    tdf = fdf[[
        "Order ID","Order Date","Ship Date","Factory",
        "State/Province","Region","Ship Mode","Product Name",
        "Division","Lead Time","Sales","Gross Profit","Is Delayed"
    ]].copy()
    tdf["Order Date"] = tdf["Order Date"].dt.strftime("%d %b %Y")
    tdf["Ship Date"]  = tdf["Ship Date"].dt.strftime("%d %b %Y")
    tdf["Is Delayed"] = tdf["Is Delayed"].map({True:"🔴 Yes", False:"🟢 No"})

    st.markdown(f"**Showing {min(500, len(tdf)):,} of {len(tdf):,} filtered records**")
    st.dataframe(tdf.head(500), use_container_width=True, height=500)

    st.markdown("### 📊 Summary Statistics")
    col1, col2, col3 = st.columns(3)
    with col1:
        st.markdown("**Lead Time Stats**")
        st.dataframe(fdf["Lead Time"].describe().round(1).to_frame(), use_container_width=True)
    with col2:
        st.markdown("**Sales Stats**")
        st.dataframe(fdf["Sales"].describe().round(2).to_frame(), use_container_width=True)
    with col3:
        st.markdown("**By Division**")
        div_sum = fdf.groupby("Division")[["Sales","Gross Profit"]].sum().round(0)
        st.dataframe(div_sum, use_container_width=True)

Writing /content/app.py


In [58]:
!pip install pyngrok -q

from pyngrok import ngrok
import subprocess, threading, time

# Kill any previous streamlit
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(2)

# Start Streamlit
def run():
    subprocess.run([
        "streamlit", "run", "/content/app.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ])

t = threading.Thread(target=run)
t.daemon = True
t.start()
time.sleep(12)

# ⬇️ PASTE YOUR OWN TOKEN HERE ⬇️
ngrok.kill()
ngrok.set_auth_token("3BveMihE5ASvXvCqtWtUurOfPw8_6eZNX4DvDd4ALHA9BRAuV")
public_url = ngrok.connect(8501)

print("=" * 60)
print("🍬  NASSAU CANDY DASHBOARD IS LIVE!")
print(f"🔗  {public_url}")
print("=" * 60)
print("👆  Click the link above to open your dashboard!")

🍬  NASSAU CANDY DASHBOARD IS LIVE!
🔗  NgrokTunnel: "https://anaya-sliceable-linwood.ngrok-free.dev" -> "http://localhost:8501"
👆  Click the link above to open your dashboard!
